# Điền topic thiếu theo tần suất trong description và README

- Topic thiếu = cột `topics` rỗng `[]`.
- Mỗi topic được chấm điểm theo tần suất tên topic và alias trong `description` (trọng số cao hơn) và README.
- README được lấy qua GitHub REST API (`GET /repos/{owner}/{repo}/readme`), vì cột `read_me` của bạn là boolean.
- Kiểm chứng trên một mẫu repo đã có topic: ẩn topic, đoán lại, đo precision@k và recall@k.

**Cần có:** `GITHUB_TOKEN` trong môi trường. Dữ liệu đầu vào: file CSV có `full_name`, `description`, `topics`.

In [1]:
import base64
import json
import math
import os
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from tqdm import tqdm

ROOT = Path("..").resolve()
HERE = Path(".").resolve()

INPUT = ROOT / "data" / "raw" / "github_repos_complete.csv"
ALIAS_FILE = HERE / "output" / "alias_accepted.json"       # tùy chọn: alias đã duyệt
README_CACHE = HERE / "output" / "readme_cache.jsonl"
OUTPUT = HERE / "output" / "topics_filled.csv"

MIN_TOPIC_REPOS = 20      # topic cần có ít nhất số repo có topic này để đưa vào danh mục
MAX_TOPICS = 4            # số topic tối đa cho mỗi repo
MIN_TOPICS = 3            # dưới ngưỡng này thì đánh dấu low_confidence
MIN_SCORE = 0.5           # điểm tối thiểu để giữ topic
W_DESC = 2.0              # trọng số xuất hiện trong description
EVAL_SAMPLE = 100         # số repo có topic dùng để kiểm chứng
REQUEST_SLEEP = 0.05
SEED = 42

In [2]:
def make_session():
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise SystemExit("Thiếu biến môi trường GITHUB_TOKEN")
    s = requests.Session()
    s.headers.update({
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {token}",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "topic-filler",
    })
    return s


def fetch_readme(session, full_name, retries=3):
    """Trả về nội dung README, hoặc '' nếu repo không có README."""
    url = f"https://api.github.com/repos/{full_name}/readme"
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=30)
        except requests.RequestException:
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 200:
            data = r.json()
            if data.get("encoding") != "base64":
                return ""
            return base64.b64decode(data["content"]).decode("utf-8", errors="replace")
        if r.status_code == 404:
            return ""
        if r.status_code in (403, 429):
            reset = r.headers.get("X-RateLimit-Reset")
            wait = max(int(reset) - int(time.time()), 0) + 5 if reset else 60
            print(f"Bị giới hạn, chờ {wait} giây")
            time.sleep(min(wait, 3600))
            continue
        time.sleep(2 ** attempt)
    return ""


def load_readme_cache():
    cache = {}
    if README_CACHE.exists():
        with open(README_CACHE, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    cache[rec["full_name"]] = rec["read_me"]
    return cache


def readmes_for(names, session, cache):
    todo = [n for n in names if n not in cache]
    if todo:
        README_CACHE.parent.mkdir(parents=True, exist_ok=True)
        with open(README_CACHE, "a", encoding="utf-8") as f:
            for n in tqdm(todo, desc="README"):
                text = fetch_readme(session, n)
                cache[n] = text
                f.write(json.dumps({"full_name": n, "read_me": text}, ensure_ascii=False) + "\n")
                f.flush()
                time.sleep(REQUEST_SLEEP)
    return {n: cache[n] for n in names}

In [3]:
CODE_BLOCK = re.compile(r"```.*?```", re.S)
INLINE_CODE = re.compile(r"`[^`]*`")
IMAGE = re.compile(r"!\[[^\]]*\]\([^)]*\)")
MD_LINK = re.compile(r"\[([^\]]*)\]\([^)]*\)")
URL = re.compile(r"https?://\S+|www\.\S+")
HTML_TAG = re.compile(r"<[^>]+>")


def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = CODE_BLOCK.sub(" ", t)
    t = INLINE_CODE.sub(" ", t)
    t = IMAGE.sub(" ", t)
    t = MD_LINK.sub(r"\1", t)
    t = URL.sub(" ", t)
    t = HTML_TAG.sub(" ", t)
    t = t.lower()
    t = re.sub(r"[_\-/|*#>~=\[\]\(\)\{\}:;,!?\"'\\.]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def ngram_counts(text, n_max=3):
    toks = clean_text(text).split()
    c = Counter()
    for n in range(1, n_max + 1):
        for i in range(len(toks) - n + 1):
            c[" ".join(toks[i:i + n])] += 1
    return c, len(toks)


def phrases_of(topic, aliases):
    """Tên topic và các alias, đã chuẩn hóa cùng cách với văn bản."""
    out = {clean_text(topic.replace("-", " "))}
    for a in aliases.get(topic, []):
        out.add(clean_text(a))
    return {p for p in out if p}


def to_list(v):
    if isinstance(v, str) and v.strip().startswith("["):
        return sorted({str(x).strip().lower() for x in json.loads(v) if str(x).strip()})
    return []

## 1. Đọc dữ liệu và chia nhóm

In [4]:
raw = pd.read_csv(INPUT, dtype=str, keep_default_na=False, encoding="utf-8-sig")
raw["full_name"] = raw["full_name"].str.strip()
raw["description"] = raw["description"].fillna("").astype(str)
raw["topic_list"] = raw["topics"].map(to_list)

labeled = raw[raw["topic_list"].map(len) > 0].reset_index(drop=True)
missing = raw[raw["topic_list"].map(len) == 0].reset_index(drop=True)
print(f"Có topic: {len(labeled)} | Thiếu topic: {len(missing)}")

aliases = json.loads(ALIAS_FILE.read_text(encoding="utf-8")) if ALIAS_FILE.exists() else {}
print(f"Alias đã duyệt: {sum(len(v) for v in aliases.values())} cho {len(aliases)} topic")

Có topic: 7248 | Thiếu topic: 4752
Alias đã duyệt: 0 cho 0 topic


## 2. Chấm điểm và chọn topic

In [5]:
def build_vocab(repos):
    """Topic đủ phổ biến và số repo có topic đó."""
    counts = Counter(t for lst in repos["topic_list"] for t in lst)
    return {t: c for t, c in counts.items() if c >= MIN_TOPIC_REPOS}


def score_text(desc, readme, vocab, topic_df, N):
    d_counts, _ = ngram_counts(desc)
    r_counts, n_tok = ngram_counts(readme)
    length_norm = 1 + n_tok / 200
    scores = {}
    for topic in vocab:
        ps = phrases_of(topic, aliases)
        c_desc = sum(d_counts[p] for p in ps)
        c_read = sum(r_counts[p] for p in ps)
        if c_desc + c_read == 0:
            continue
        tf = (W_DESC * c_desc + c_read) / length_norm
        idf = math.log(1 + N / topic_df[topic])
        scores[topic] = tf * idf
    return scores


def pick(scores):
    top = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))
    chosen = [(t, s) for t, s in top[:MAX_TOPICS] if s >= MIN_SCORE]
    return chosen


def precision_recall(pred, true):
    if not pred:
        return 0.0, 0.0
    hit = len(set(pred) & set(true))
    return hit / len(pred), hit / len(true) if true else 0.0

## 3. Kiểm chứng trên repo đã có topic

In [6]:
session = make_session()
cache = load_readme_cache()

rng = pd.Series(range(len(labeled))).sample(min(EVAL_SAMPLE, len(labeled)), random_state=SEED).tolist()
eval_df = labeled.iloc[rng].reset_index(drop=True)
train_df = labeled.drop(index=rng).reset_index(drop=True)

vocab_train = build_vocab(train_df)
topic_df_train = {t: vocab_train[t] for t in vocab_train}
N_train = len(train_df)
print(f"Danh mục topic (từ dữ liệu huấn luyện): {len(vocab_train)} topic")

eval_readmes = readmes_for(eval_df["full_name"].tolist(), session, cache)
p_at, r_at = [], []
for _, row in eval_df.iterrows():
    scores = score_text(row["description"], eval_readmes[row["full_name"]], vocab_train,
                        topic_df_train, N_train)
    pred = [t for t, _ in pick(scores)]
    true = [t for t in row["topic_list"] if t in vocab_train]
    if not true:
        continue
    p, r = precision_recall(pred, true)
    p_at.append(p)
    r_at.append(r)

print(f"Trên {len(p_at)} repo kiểm chứng:")
print(f"  precision@{MAX_TOPICS} (trung bình): {sum(p_at) / len(p_at):.3f}")
print(f"  recall@{MAX_TOPICS} (trung bình):    {sum(r_at) / len(r_at):.3f}")

Danh mục topic (từ dữ liệu huấn luyện): 430 topic


README: 100%|██████████| 100/100 [00:49<00:00,  2.02it/s]


Trên 90 repo kiểm chứng:
  precision@4 (trung bình): 0.243
  recall@4 (trung bình):    0.239


## 4. Điền topic cho repo thiếu

In [7]:
vocab = build_vocab(labeled)
topic_df = {t: vocab[t] for t in vocab}
N = len(labeled)
print(f"Danh mục đầy đủ: {len(vocab)} topic")

missing_readmes = readmes_for(missing["full_name"].tolist(), session, cache)
rows = []
for _, row in tqdm(missing.iterrows(), total=len(missing), desc="Chấm điểm"):
    scores = score_text(row["description"], missing_readmes[row["full_name"]], vocab, topic_df, N)
    chosen = pick(scores)
    rows.append({
        "full_name": row["full_name"],
        "topics_filled": json.dumps([t for t, _ in chosen], ensure_ascii=False),
        "scores": json.dumps({t: round(s, 3) for t, s in chosen}, ensure_ascii=False),
        "n_topics": len(chosen),
        "low_confidence": len(chosen) < MIN_TOPICS,
        "topics_imputed": len(chosen) > 0,
    })

result = pd.DataFrame(rows)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
print(f"Đã điền {int(result['topics_imputed'].sum())}/{len(result)} repo, "
      f"{int(result['low_confidence'].sum())} repo dưới {MIN_TOPICS} topic")
print(f"Lưu tại {OUTPUT}")
result.head(10)

Danh mục đầy đủ: 434 topic


README:  92%|█████████▏| 4393/4752 [36:19<02:27,  2.43it/s]   

Bị giới hạn, chờ 476 giây


Chấm điểm: 100%|██████████| 4752/4752 [00:23<00:00, 202.13it/s]

Đã điền 4621/4752 repo, 331 repo dưới 3 topic
Lưu tại D:\HomeWork\IE313\Đồ án\analysis-popularity-github-repository\notebooks\output\topics_filled.csv


,full_name,topics_filled,scores,n_topics,low_confidence,topics_imputed
0,datalab-to/marker,"[""server"", ""vlm"", ""ocr"", ""gpu""]","{""server"": 7.178, ""vlm"": 6.995, ""ocr"": 6.021, ...",4,False,True
1,Genesis-Embodied-AI/genesis-world,"[""gui"", ""robotics"", ""github"", ""embodied-ai""]","{""gui"": 6.332, ""robotics"": 3.459, ""github"": 2....",4,False,True
2,deepseek-ai/DeepSeek-Coder,"[""deepseek"", ""data"", ""evaluation"", ""performance""]","{""deepseek"": 12.336, ""data"": 7.934, ""evaluatio...",4,False,True
3,ZachGoldberg/Startup-CTO-Handbook,"[""software-engineering"", ""markdown"", ""email"", ...","{""software-engineering"": 6.474, ""markdown"": 4....",4,False,True
4,zai-org/ChatGLM3,"[""api"", ""huggingface"", ""chat"", ""langchain""]","{""api"": 12.302, ""huggingface"": 10.145, ""chat"":...",4,False,True
5,Vaibhavs10/insanely-fast-whisper,"[""whisper"", ""transformers"", ""cli"", ""mac""]","{""whisper"": 13.018, ""transformers"": 11.516, ""c...",4,False,True
6,dotnet/eShop,"[""data"", ""cli"", ""sdk"", ""c""]","{""data"": 4.038, ""cli"": 3.34, ""sdk"": 2.971, ""c""...",4,False,True
7,Melkeydev/go-blueprint,"[""go"", ""database"", ""ui"", ""websocket""]","{""go"": 17.201, ""database"": 11.438, ""ui"": 5.496...",4,False,True
8,apple-aiml-research/ml-ferret,"[""ui"", ""evaluation"", ""web"", ""apple""]","{""ui"": 4.377, ""evaluation"": 4.076, ""web"": 3.56...",4,False,True
9,OpenBMB/XAgent,"[""data"", ""data-analysis"", ""tools"", ""agent""]","{""data"": 5.169, ""data-analysis"": 4.026, ""tools...",4,False,True


**Lưu ý:** `topics = []` có thể là repo không có topic thật. Điền topic cho nhóm này là giả định rằng
trạng thái rỗng là lỗi thu thập. Nếu không chắc, hãy xem cột `low_confidence` và kiểm tra thủ công trước khi dùng.